# 08 — Choosing the Whisper model

The application transcribes with Whisper **base**. Notebook 04 found that on the
questions whose clip really answers them, retrieval barely suffers from recognition
errors but the answers do: Whisper misspells the very words the answer is made of.
This notebook compares **base**, **small** and **turbo** (large-v3-turbo) on the same
300 SLUE-SQA-5 questions, against the ceiling set by the reference transcript.

**Re-running.** Each model is a run of notebook 04 with `WHISPER` set to `"base"`,
`"small"` or `"turbo"`. Transcription runs on the CPU, as in the application; the
times below include loading the model once per recording, as an upload does.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

In [3]:
import random

runs = {"base": saved("spoken_qa_300q.json"),
        "small": saved("spoken_qa_300q_whisper-small.json"),
        "turbo": saved("spoken_qa_300q_whisper-turbo.json")}
labels = saved("spoken_qa_support_labels.json")["labels"]

rows = []
for model, run in runs.items():
    clean = run["results"]["whisper"]["by_support"]["answered_by_clip"]
    rows.append({"model": model, "times real time": run["realtime_factor"],
                 "transcribe minutes": round(run["transcribe_seconds"] / 60, 1),
                 "answer in transcript (all 300)": run["results"]["whisper"]["answer_in_transcript"],
                 **{m: clean[m] for m in ("recall@3", "answer_in_context", "exact_match", "f1")}})
reference = runs["base"]["results"]["reference"]["by_support"]["answered_by_clip"]
rows.append({"model": "reference transcript",
             **{m: reference[m] for m in ("recall@3", "answer_in_context", "exact_match", "f1")}})
print(f"{reference['n']} questions whose clip answers them")
pd.DataFrame(rows).set_index("model").round(3)

66 questions whose clip answers them


,times real time,transcribe minutes,answer in transcript (all 300),recall@3,answer_in_context,exact_match,f1
model,,,,,,,
base,20.1,8.7,0.860,0.924,0.697,0.303,0.376
small,6.0,29.0,0.883,0.924,0.758,0.364,0.434
turbo,2.1,83.7,0.900,0.924,0.803,0.333,0.430
reference transcript,NaN,NaN,NaN,0.955,0.955,0.455,0.521


### Is the difference real?

Paired over the same clean questions: the change in F1 per question, with a 95% bootstrap interval (10,000 resamples).

In [4]:
per_question = {m: {r["question_id"]: r["f1"] for r in run["records"]["whisper"]}
                for m, run in runs.items()}
clean_ids = [q for q in per_question["base"] if labels[q]["label"] == "yes"]
rng = random.Random(0)
rows = []
for a, b in (("base", "small"), ("base", "turbo"), ("small", "turbo")):
    diff = [per_question[b][q] - per_question[a][q] for q in clean_ids]
    means = sorted(sum(rng.choice(diff) for _ in diff) / len(diff) for _ in range(10000))
    rows.append({"from": a, "to": b, "change in F1": round(sum(diff) / len(diff), 3),
                 "95% interval": f"[{means[250]:+.3f}, {means[9750]:+.3f}]",
                 "better": sum(d > 0 for d in diff), "worse": sum(d < 0 for d in diff)})
pd.DataFrame(rows)

,from,to,change in F1,95% interval,better,worse
0,base,small,0.058,"[-0.047, +0.165]",12,9
1,base,turbo,0.054,"[-0.034, +0.146]",12,9
2,small,turbo,-0.004,"[-0.108, +0.101]",11,11
